<a href="https://colab.research.google.com/github/Mechanics-Mechatronics-and-Robotics/CV-2026/blob/main/Week_04/Lab4_Generative_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4. Generative models (VAE, GAN, Diffusion models).


## Motivation and theoretical basis

**Generative artificial intelligence (generative AI, GenAI)** is a subset of artificial intelligence that uses generative models to produce text, images, videos, or other forms of data.

A generative model is a statistical model of [the joint probability distribution](https://en.wikipedia.org/wiki/Joint_probability_distribution) P(X,Y) on a given observable variable X and target variable Y. A generative model can be used to "generate" random instances of an observation x. It also describes models that generate instances of output variables in a way that has no clear relationship to probability distributions over potential samples of input variables.

In ML we are usually tied with two types of models:

**a discriminative model** - is a model of the conditional probability of the target Y, given an observation x, symbolically,  P(Y∣X=x) (e.g. supervised learning)

**a generative model** is a model of the conditional probability of the observable X, given a target y, symbolically, P(X∣Y=y)

Simply saying comparing with a discriminative approach in CV, where we usually want to know what object is depicted on the image (label), for generative approach we want get a sample itself given a label.

**Fundamental problem**

Our hypothetical model from the Bayes perspective:

$P(Y|X) = \frac{P(X|Y)P(Y)}{P(X)} - \textrm{our initial hypothesis from Bayes perspective}$

$P(M|D) \sim P(D|M)P(M) - \textrm{presumably our best model}$

Learning a specific model, M, that best accounts for all the data is accomplished by **maximizing the posterior distribution** over the models. Oftentimes though we are agnostic in the prior over the model, and so we may simply choose the model that maximizes the likelihood, $P(D|M)$.

$P(D|M) = P(D_{1}|M) * P(D_{2}|M) ... P(D|M)$,

\begin{equation*}
  P(D|M) = \prod_{i=1}^{}P(D_{i}|M),
\end{equation*}

where Di denotes
an individual data item (e.g., a particular image). The probability of an individual data item is obtained by summing over all of the possible causes for the data.

IOW, we want to approximate our distribution using idea of latent represenataion to generate something new, but meaningful and could follow our initial distribution that we don't know how does it look. **Yeah, I know that it sounds ridiculous.**








### **Variational Autoencoder (VAE)**

Original Paper: https://arxiv.org/pdf/1312.6114

In basic autoencoders our initial idea is to find mostly coherent latent representation of the original data (e.g. images). Unfortenately, it is **almost impossible**... Why? **Latent space** is disorganized and irregular meaning large areas of it does not make sense for us.

Variational Autoencoders in some sense trying to organize our latent space and simply based on two simple concepts:

- **a marginal distribution**

- **a conditional distribution**

Our goal is to generate data from a given distribution p(x) - our dataset of images, we do not know it is exact shape -> we take a latent distribution p(z).
How to connect them? We present two concepts:

- a posterior distribution p(z|x) - prob. that a latent vector z was generated by an image x.

- a likelihood distribution p(x|z) - prob. of reconstructng image from latent space. This means we'll basically generate new samples from our original distribution, BUT we do not know latent distribution either and assume it as a normal distribution $N(0,1)$, which allows us to compute p(x|z).

The problem remains is how to construct p(z|x). This is where **VAE** comes into play. What does it make?

First, we approximate our posterior p(z|x) ~ q(z|x) and simply reduce the problem to find distribution parameters - $N(\mu, \sigma)$. This is an optimization process usually called **Variational Bayes**.

Second, we can decoder to reconstruct our images based on these latent parameters. The whole process can be senn on the image below

<img src="https://i.postimg.cc/9Q8cxV57/VAE.png" width="400" height="100">

How do we train such architecture? This is our objectve function.

$L(X) = E_{q(x|z)}[logp(x|z)] - KL(q(z|x) | p(z))$ - Evidence Lower Bound

- first term is called **a data consistency** and it shows how well our model can reconstruct our image from encoded version of z - just take an image, encode it and calculate a loss of recostruction (just like in basic AEs).

- second one is **a regularization term**. As you might notice we are using Kullback-Leibler divergence here, which measures distance between two probability distributions. In our case it shows how close our posterior p(z|x) to a prior distribution p(z), which is normal in our case. By this we limit our approximated posterior distribution to take a shape of normal distribution as well.

Let's sumamrize the whole algorithm:

1.   The encoder mapps our images into the probability distribution, which is Gaussian in our case by transforming data into parameters.
2.   From this latent distribution we randomly choose a point and converts it back to an input space.
3. Caculate ELBO loss and backpropagate it through the network

$L(x,x{'}) = L_{2}(x,x{'}) - D_{KL}(N(\mu, \sigma) | N(0,1))$

BUT: how do we do that through a sampling process? *We can't*....

**Reparametrization Trick**

Instead of sampling directly from Gaussian we introduce an additional variable **$\epsilon$** and doing the following:

1. Choose a random point from a standard normal distribution - our $\epsilon \sim N(0,1)$
2. Then we scale it by a variance of our posterior distribution $\epsilon \cdot \sigma$
3. Finally we just shift it by its mean - $\mu + \epsilon \cdot \sigma$

Basically, it is the same as we would take a sample from our posterior distribution, but reparametrization trick makes it differentiable with respect to mean and variance.

<img src="https://i.postimg.cc/BQdsVMkp/VAE-2.png" width="600" height="200">

What does VAE give us?

- accurately represent our image space in some latent parametric distribution
- generate entirely new samples that look different from original images
- obtain some image abstractions that look as a mix of images

What are limitations?

- usually you cannot obtain high quality images

<img src="https://i.postimg.cc/zDQS4gjh/celeba-vae.png" width="600" height="200">

- we cannot generate a specific image

### **Generative Adversarial Networks (GAN)**

Original Paper: https://arxiv.org/pdf/1406.2661

Adversarial machine learning is a technique used in machine learning (ML) to fool or misguide a model with malicious input. While adversarial machine learning can be used in a variety of applications, this technique is most commonly used to execute an attack or cause a malfunction in a machine learning system.
Malicious actors carry out adversarial attacks on ML models. They have various motivations and a range of tactics. However, their aim is to negatively impact the model's performance, so it misclassifies data or makes faulty predictions. To do this, attackers either manipulate the system's input data or directly tamper with the model's inner workings.

<img src="https://i.postimg.cc/Z5pGWZnz/Screenshot-2026-02-09-at-02-26-42.png" width="500" height="300">

Let's translate it to a generation problem...

**Our goal:** to discover (to model) the underlying distribution behind a certain data generating process.

As adversarial logic suggests, let's take a discriminator and a generator, where prior is responsible for distinguishing “malicious” samples, while the other for generating “plausible” objects that could be able to fool the discriminator. Presumably by this approach, our distribution is discovered through an adversarial competition between a generator and a discriminator. Presumably we can get a robust generator by exploiting an idea of adversarial training. The whole approach is called **Generative Adversarial Nets.** GAN can be seen as an interplay between two different models: the generator and the discriminator.

Let's explain the whole process:

1. The process starts with two neural networks: the Generator Network (G) and the Discriminator Network (D). The Generator takes a random seed or noise vector as input and produces generated samples. The Discriminator takes either real data samples or generated samples as input and classifies them as real or fake.
2. A random noise vector is fed into the Generator Network. The Generator processes this noise and outputs generated data samples that are intended to resemble real data.
3. Then it generates fake data from input random noise and calculates the generator's loss using the discriminator's output. Finally, it updates the generator's weights to minimize the loss.
4. It takes a batch of real data and a batch of fake data. Then it calculates the discriminator's loss for both real and fake data. Finally, it updates the discriminator's weights to minimize the loss.
5. Repeat steps 2 to 4. During each iteration, both the Generator and Discriminator are alternately trained and try to improve each other's performance. This alternating optimization continues until the generator generates data that is identical to the real data and the discriminator can no longer reliably distinguish between real and fake data. IOW, we play in a min-max game.


<img src="https://i.postimg.cc/SsvTZtRW/GAN-training.png" width="500" height="300">

HOW to translate it into a loss function?

The original formulation presents it as a maximization problem for LD, with the sign obviously flipped. Then, authors proceed this with framing it as a min-max game, where the discriminator seeks to maximize the given quantity whereas the generator seeks to achieve the reverse. In other words                                                                           


<img src="https://i.postimg.cc/c19x2pbK/gan_loss_1.png" width="500" height="50">

OR

<img src="https://i.postimg.cc/FFWr8MBd/gan_loss_2.png" width="700" height="50">

Practically, we apply separate loss functions, as maximizing log(D(x)) happens quicker than than minimizing log(1-D(z)). This insists to update gradient for discriminator only once for several step of generator.


### **Diffusion models**

Original Paper: https://arxiv.org/pdf/2006.11239

Remember that our main problem - we don't know our real distriubtion p(x) for image data, but even without explicit formula we still want to generate some new samples from it. Challenge is to find a way to create new samples without having good way of describing **underlying distribution**.

Central idea in diffusion models - overcome this problem by **removing Gaussian noise from image**. Briefly this is what happens:

Basically, we gradually corrupt our images by adding Gaussian noise. At each step a small amout of noise is added slowly erasing the structure of the image. By continuing this process we will ended up with a pure noise.  

<img src="https://i.postimg.cc/J7PYKGXg/diffusion-training.png" width="600" height="300">

The key idea to train then a model that learns to slowly reverse these steps and remove noise bit-by-bit. In the case of success model should be able to start from a random noise and refine it into a meaningful image. Basically, our goal is to transforms our data into normal distribution.

Let's consider all steps in detail.

**Initial idea (we can skip it...)**

1. start with an image without noise $x_{0}$
2. to generate next image $x_{1}$  we use a conditional distribution

$q(x_{1}|x_{0})$, we produce $x_{1} = x_{0} + \beta \cdot \epsilon, \epsilon \sim N(0,1)$.

It means that actual distribution describing $q(x_{1}|x_{0}) = N(x_{0}, \beta).$ This transforms our problem from "adding noise" to $x_{0}$ to "taking a sample from Gaussian" with variance $\beta$ and centered to $x_{0}$.

3. Next step we label distriubtion as $q(x_{2}|x_{1})$ and repeat the process, which is

$x_{2} = x_{1} + \beta \cdot \epsilon$ and again which is the same as taking sample from Gaussian distribution

$q(x_{2}|x_{1}) = N(x_{1}, \beta)$

4. Process is about just adding noise meaning we can go straight from $x_{0}$ to $x_{2}$ just by adding twice as much noise and change our formula to

$x_{2} = x_{0} + 2\beta \cdot \epsilon$ and our distriubution will look like this:

$q(x_{2}|x_{0}) = N(x_{0}, 2\beta)$

5. We keep repeating this process until we reach the timestep T. This means at any given step what we are doing is just adding $t\beta$ times Gaussian noise to $x_{0}$.

$x_{t} = x_{0} + t\beta \cdot \epsilon$

$q(x_{t}|x_{0}) = N(x_{0}, t\beta)$

Problem? This is not leading us to Normal distribution, as **variance explodes.**

<img src="https://i.postimg.cc/qR8SkT1w/diffusion-3.png" width="500" height="300">

**How does it work?**

We want that $q(x_{t}|x_{0}) \rightarrow N(0,1)$. How to get these parameters?
Let's modify our process by adding a term to our sample formualation

$q(x_{t}|q_{x-1}) = \sqrt{1-\beta}x_{t-1}+\beta \cdot \epsilon$, which was derived from

$q(x_{t}|x_{0}) = \sqrt{\overline{\alpha}}x_{0} + (1- \overline{\alpha}) \cdot \epsilon$, where

$\overline{\alpha} = (1-\beta)^{t}$

This formula will help us to slowly transform our p(x) to Normal distribution.
However, given that $\beta$ varies at each step we should change our formula to

\begin{equation*}
  \overline{\alpha}_{t} = \prod_{i=1}^{t} (1- \beta_{t})
\end{equation*}

<img src="https://i.postimg.cc/L6VFMtzZ/noise.png" width="300" height="50">


This can be considered a defined diffusion process. After we know how to transform our image into noise, we need to define a process how to reverse it back to our initial image $p_{\theta}(x_{t-1})|x_{t})$.

This is about finding right parameters $\theta$, which parameters of our neural network. How do we train them? Basically it is about minimizing **the negative log likelihood** ($-log(p_{\theta}(x_{t-1})|x_{t})$).

<img src="https://i.postimg.cc/T3vByhD6/diffusion-4.png" width="500" height="200">

If we decompose our noising process it is basically sets of conditional distributions.

$q(x_{1},...,x_{T} | x_{0}) = q(x_{1} | x_{0}) q(x_{2}|x_{1},x_{0}) …q(x_{T}|x_{T-1},...,x_{0})$

\begin{equation*}
  q(x_{1:T}|x_{0}) = \prod_{i=1}^{T}q(x_{t}|x_{t-1}) - \textrm{forward process}
\end{equation*}

\begin{equation*}
  p_{\theta}(x_{0:T}) = p_{\theta}(x_{T})\prod_{i=1}^{T}p_{\theta}(x_{t-1}|x_{t}) - \textrm{reverse process}
\end{equation*}

Now how does look like our objective function?

<img src="https://i.postimg.cc/d3r3BLQR/obj-func.png" width="300" height="50">

It is intractable meaning we can switch to **ELBO**.

We skip the process of derivation, but in our case it is just simply minimizing the sum of of KL divergences in our objective function bringing its approximate posterior to its counterpart.

<img src="https://i.postimg.cc/FFDBrD91/obj-func-2.png" width="w00" height="200">

On the image below you can see how we obtain our final of objective function.

<img src="https://i.postimg.cc/TwZJk7g1/diffusion-obj-func.png" width="900" height="450">



## Practical part

In [ ]:
# prerequisites
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
from torchvision.utils import save_image, make_grid
from tqdm import tqdm_notebook as tqdm

%matplotlib inline
from matplotlib import pyplot as plt
from IPython.display import clear_output
# set inline plots size
plt.rcParams["figure.figsize"] = (16, 10) # (w, h)
# remove grid lines
import numpy as np
import cv2
import os

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# helper functions
def mkdir(path):
    if not os.path.exists(path): os.makedirs(path)

def show_in_row(list_of_images, titles = None, disable_ticks = False):
  count = len(list_of_images)
  for idx in range(count):
    subplot = plt.subplot(1, count, idx+1)
    if titles is not None:
      subplot.set_title(titles[idx])

    img = list_of_images[idx]
    cmap = 'gray' if (len(img.shape) == 2 or img.shape[2] == 1) else None
    subplot.imshow(img, cmap=cmap)
    if disable_ticks:
      plt.xticks([]), plt.yticks([])
  plt.show()

### GAN
![alt text](https://i.postimg.cc/sxKS57hR/GAN.png)

In [ ]:
bs = 100

# MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

train_dataset = datasets.MNIST(root='./mnist_data/', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./mnist_data/', train=False, transform=transform, download=False)

# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=bs, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=bs, shuffle=False)

In [ ]:
class Generator(nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()
        self.fc1 = nn.Linear(g_input_dim, 256)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features*2)
        self.fc3 = nn.Linear(self.fc2.out_features, self.fc2.out_features*2)
        self.fc4 = nn.Linear(self.fc3.out_features, g_output_dim)

    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.leaky_relu(self.fc3(x), 0.2)
        return torch.tanh(self.fc4(x))

class Discriminator(nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, 1024)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = nn.Linear(self.fc2.out_features, self.fc2.out_features//2)
        self.fc4 = nn.Linear(self.fc3.out_features, 1)

    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc3(x), 0.2)
        x = F.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))

In [ ]:
# train_dataset.train_data.size()

In [ ]:
# build network
z_dim = 100
mnist_dim = train_dataset.train_data.size(1) * train_dataset.train_data.size(2)

G = Generator(g_input_dim = z_dim, g_output_dim = mnist_dim).to(device)
D = Discriminator(mnist_dim).to(device)

In [ ]:
G

In [ ]:
D

In [ ]:
# loss
criterion = nn.BCELoss()

# optimizer
lr = 0.0002
G_optimizer = optim.Adam(G.parameters(), lr = lr)
D_optimizer = optim.Adam(D.parameters(), lr = lr)

In [ ]:
def D_train(x):
    #=======================Train the discriminator=======================#
    D.zero_grad()

    # train discriminator on real
    x_real, y_real = x.view(-1, mnist_dim), torch.ones(bs, 1)
    x_real, y_real = Variable(x_real.to(device)), Variable(y_real.to(device))

    D_output = D(x_real)
    D_real_loss = criterion(D_output, y_real)
    D_real_score = D_output

    # train discriminator on fake
    z = Variable(torch.randn(bs, z_dim).to(device))
    x_fake, y_fake = G(z), Variable(torch.zeros(bs, 1).to(device))

    D_output = D(x_fake)
    D_fake_loss = criterion(D_output, y_fake)
    D_fake_score = D_output

    # gradient backprop & optimize ONLY D's parameters
    D_loss = D_real_loss + D_fake_loss
    D_loss.backward()
    D_optimizer.step()

    return  D_loss.data.item()

In [ ]:
def G_train(x):
    #=======================Train the generator=======================#
    G.zero_grad()

    z = Variable(torch.randn(bs, z_dim).to(device))
    y = Variable(torch.ones(bs, 1).to(device))

    G_output = G(z)
    D_output = D(G_output)
    G_loss = criterion(D_output, y)

    # gradient backprop & optimize ONLY G's parameters
    G_loss.backward()
    G_optimizer.step()

    return G_loss.data.item()

In [ ]:
n_epoch = 10
save_every = 1
mkdir('weights')

for epoch in tqdm(range(1, n_epoch+1), desc="Epochs"):
    D_losses, G_losses = [], []

    for batch_idx, (x, _) in enumerate(train_loader):
        D_losses.append(D_train(x))
        G_losses.append(G_train(x))

    print(G.state_dict())

    torch.save(G.state_dict(), "weights/G_{:03d}.pth".format(epoch))
    torch.save(D.state_dict(), "weights/D_{:03d}.pth".format(epoch))
    # for load
    # G.load_state_dict(torch.load(path))
    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f' % (
            (epoch), n_epoch, torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))))

print('We are done')

In [ ]:
def show_generator_results(batch_vectors: np.array):
  with torch.no_grad():
    test_z = Variable(torch.Tensor(batch_vectors).to(device))
    generated = G(test_z)

    grid = make_grid(generated.view(generated.size(0), 1, 28, 28)).cpu().numpy()[0]
    show_in_row([grid], disable_ticks=True)


show_generator_results(np.random.rand(64, z_dim) * 2 - 1)

In [ ]:
def generate_changing_vec(examples_count: int, vec_len: int) -> np.array:
    # return np.array of shape (examples_count, vec_len)
    vec1 = np.random.rand(1, vec_len)*2-1
    vec2 = np.random.rand(1, vec_len)*2-1

    # perform interpolation between vecs

    # everything below TODO
    vec_arr = vec2
    for i in range(1, examples_count):
        percent = float(i) / float(examples_count-1)
        print(percent)
        vec = percent * vec1 + (1-percent)*vec2
        vec_arr = np.vstack((vec_arr, vec))
    return vec_arr


a = generate_changing_vec(10, 100)

In [ ]:
with torch.no_grad():
  t = Variable(torch.Tensor(a).to(device))
  generated = G(t)

  grid = make_grid(generated.view(generated.size(0), 1, 28, 28)).cpu().numpy()[0]
  show_in_row([grid], disable_ticks=True)

### Simple Diffusion Model

In [ ]:
!pip install deepinv

In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np
from deepinv.models.diffunet import DiffUNet
import os
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set hyperparameters
batch_size = 32
num_epochs = 100
lr = 1e-4
image_size = 32  # MNIST is 28x28, but we'll resize to 32x32 for the model

transform = transforms.Compose(
    [
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=4
)

Define model

In [ ]:
model = DiffUNet(
    in_channels=1, out_channels=1, pretrained=None  # MNIST is grayscale
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)


Parameters for diffusion

In [ ]:
timesteps = 1000  # Total number of diffusion steps
beta_start = 1e-4  # Starting value for noise schedule
beta_end = 0.02  # Ending value for noise schedule

# Linear noise schedule
betas = torch.linspace(beta_start, beta_end, timesteps, device=device) # how many noise will be added
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat(
    [torch.tensor([1.0], device=device), alphas_cumprod[:-1]]
)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

Training step

In [ ]:
all_losses = []

for epoch in range(num_epochs):
    total_loss = 0.0
    epoch_losses = []  # Store losses for this epoch

    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad()

        # Sample random timesteps
        t = torch.randint(0, timesteps, (images.shape[0],), device=device)

        # Sample noise
        noise = torch.randn_like(images)

        # Apply forward diffusion process at timestep t
        noised_images = (
            sqrt_alphas_cumprod[t, None, None, None] * images
            + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
        )

        # Predict noise
        noise_pred = model(noised_images, t, type_t="timestep")

        # Calculate loss (the model predicts the noise that was added)
        loss = nn.MSELoss()(noise_pred, noise)

        loss.backward()
        optimizer.step()

        # Save the loss value
        loss_value = loss.item()
        total_loss += loss_value
        epoch_losses.append(loss_value)

torch.save(model.state_dict(), "./weights/model_final.pth")

Inference

In [ ]:
import torch
import deepinv
from pathlib import Path

device = "cuda"
image_size = 32

checkpoint_path = "./checkpoints/trained_diffusion_model.pth"
model = deepinv.models.DiffUNet(
    in_channels=1, out_channels=1, pretrained=Path(checkpoint_path)
).to(device)

beta_start = 1e-4
beta_end = 0.02
timesteps = 1000

betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

model.eval()

n_samples = 32

with torch.no_grad():
    x = torch.randn(n_samples, 1, image_size, image_size).to(device)

    for t in reversed(range(timesteps)):
        t_tensor = torch.ones(n_samples, device=device).long() * t

        predicted_noise = model(x, t_tensor, type_t="timestep")

        alpha = alphas[t]
        alpha_cumprod = alphas_cumprod[t]
        beta = betas[t]

        if t > 0:
            noise = torch.randn_like(x)
        else:
            noise = 0

        x = (1 / torch.sqrt(alpha)) * (
            x - (beta / torch.sqrt(1 - alpha_cumprod)) * predicted_noise
        ) + torch.sqrt(beta) * noise

# Hometask


## First task (for first task you can choose between several tasks!)

You need to train you own Variational Autoencoder based on any colorful dataset you could choose (e.g. CIFAR10, CelebA dataset, etc.).

**Requirements to your solution:**

1. Encoder and Decoder realizations using PyTorch framework as we did in the GAN section (basically from scratch).

2. After the training phase, you should provide at least 10 randomly generated images, which should have patterns distinguishable to the human eye. This means the generated images shouldn't be just random noise, but they also don't have to be perfectly generated samples (although this is desirable).

3. Provide comments for each step in your code.


### OR

You need to train your own GAN model for **image denoising** purpose using ANY dataset you want (colorful or grayscale).

**Requirements to your solution:**

1. You can choose any architecture you want for generator and discriminator respectively. Our example is adapted to grayscale images.

2. Patterns of noise can be chosen (blurring, crops, normally deistributed noise or salt and pepper noise, etc.). You choose one or several noise types to deal with.

3. Before training please provide examples of noised and denoised images.

5. After training phase your model should be able to overcome chosen noise patterns (not ideally).

6. Provide comments for each step in your code.

Examples:

https://github.com/fastai/course-v3/blob/master/nbs/dl1/lesson7-superres-gan.ipynb

https://lajavaness.medium.com/image-denoising-with-gan-17c3e6b97105


### OR

Train your own **diffusion model**.

**Requirements to your solution:**

1. Choose any dataset you want for a model training (colorful or grayscale - MNIST, CIFAT, FFHQ, etc.). But remember it is a very time-consuming task.
2. Show architecture and explain why you've made this choice. You can use the one provided during the lab.
3. After trainng step generate **at least** five examples from your model with distinguishable patterns.
4. Provide comments for each step in your code.
5. Make your own solution, but it could be based on pre-training models.

Examples:

https://github.com/ytdeepia/DDPM/tree/main

https://medium.com/@geronimo7/training-a-latent-diffusion-model-from-scratch-897c7b77ece9